In [ ]:
"""
FePt Hemispherical Cap — Hysteresis Loop, Pure L10 Phase, Diameter Sweep
=========================================================================
Simulates hysteresis loops for pure L10-phase FePt hemispherical caps using
OOMMF via the ubermag Python interface. All cells are assigned the full L10
anisotropy constant — no A1 soft phase. The SiO2 sphere is NOT included;
only the FePt shell is modelled.

Cap geometry
------------
Fixed cap thickness of 60 nm across all sphere diameters.
  R_inner = sphere_diameter / 2
  R_outer = R_inner + cap_thickness   (outer radius = sphere radius + film)

Anisotropy mode (selected at runtime)
--------------------------------------
  r — Radial: easy axis points along the local outward surface normal.
              Models FePt growth with c-axis following surface curvature.
  v — Vertical: easy axis fixed along global z for every cell.
                Models FePt with out-of-plane uniaxial texture.

Solver (selected at runtime via temperature)
---------------------------------------------
  T = 0 K  →  MinDriver (static energy minimisation, no dynamics)
  T > 0 K  →  TimeDriver (LLG time integration with stochastic thermal field)

Sphere diameters: 1, 3, 5, 8, 10, 20 µm | Cap thickness: 60 nm (fixed)

Inputs  : Temperature [K] and anisotropy mode entered at runtime.
          Physical parameters defined in Section 1.
Outputs : Per-cap CSV files, a combined master CSV, and an SVG summary plot —
          all saved to a timestamped output folder.
"""

# --- Standard library ---
import os
from datetime import datetime

# --- Third-party ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --- Micromagnetics stack ---
import micromagneticmodel as mm
import discretisedfield as df
import oommfc as oc


# ===========================================================================
# 1. PARAMETERS & STUDY SETUP
# ===========================================================================

# --- Runtime prompts ---
T = int(input(
    "Enter simulation temperature in Kelvin (0 = static MinDriver, >0 = dynamic TimeDriver): "
))
USE_DYNAMIC = T > 0  # True → TimeDriver + LLG dynamics; False → MinDriver

aniso_input = input("Anisotropy mode (r = Radial, v = Vertical): ").strip().lower()
while aniso_input not in ('r', 'v'):
    aniso_input = input("  Please enter 'r' for Radial or 'v' for Vertical: ").strip().lower()
mode = "Radial" if aniso_input == 'r' else "Uniaxial_Vertical"

# --- LLG dynamics parameters (only used when T > 0) ---
alpha    = 0.05   # Gilbert damping constant (dimensionless)
run_time = 1e-10  # Integration time per field step [s]

# --- Sphere diameters to simulate [m] ---
diameters = [1e-6, 3e-6, 5e-6, 8e-6, 10e-6, 20e-6]

# --- Material parameters (pure L10 FePt — no A1 soft phase) ---
Ms = 1.0e6    # Saturation magnetisation [A/m]
A  = 1.0e-11  # Exchange stiffness [J/m]
Ku = 6.6e6    # Uniaxial anisotropy constant — L10 phase [J/m³]

# --- Field sweep parameters ---
mu0    = 4 * np.pi * 1e-7  # Vacuum permeability [H/m]
B_max  = 18.0               # Maximum applied field magnitude [T]
B_tilt = 0.02               # Small in-plane tilt to break symmetry and avoid saddle points [T]

# --- Geometry ---
cap_thickness = 60e-9  # FePt cap thickness [m] — fixed for this study

# --- Output ---
timestamp     = datetime.now().strftime("%Y-%m-%d_%H-%M")
T_label       = f"T{T}K" if USE_DYNAMIC else "0K_static"
OUTPUT_FOLDER = f"L10_DiameterSweep_{mode}_{T_label}_{timestamp}"
os.makedirs(OUTPUT_FOLDER, exist_ok=True)

# Full field sweep: high → low → high (242 steps total)
B_fields = np.concatenate([
    np.linspace(B_max, -B_max, 121),
    np.linspace(-B_max,  B_max, 121)
])

successful_runs = {}  # Collects {run_name: Mz/Ms array} for the final export

solver_label = f"TimeDriver @ {T} K" if USE_DYNAMIC else "MinDriver (0 K)"
print(f"\n--- Pure L10 FePt | {mode} | {solver_label} ---")
print(f"Output folder: {OUTPUT_FOLDER}\n")


# ===========================================================================
# 2. MAIN EXECUTION LOOP
# ===========================================================================

for d_idx, sphere_diameter in enumerate(diameters):

    size_um    = sphere_diameter * 1e6
    safe_name  = f"cap_{str(size_um).replace('.', '_')}um_{mode}_L10"
    checkpoint = os.path.join(OUTPUT_FOLDER, f"data_{safe_name}_{timestamp}.csv")

    # --- Resume support: skip if this run already has saved data ---
    if os.path.exists(checkpoint):
        print(f">> Skipping {safe_name} — data already exists.")
        existing = pd.read_csv(checkpoint)
        successful_runs[safe_name] = existing['Mz/Ms'].values
        continue

    # --- Geometry: hemispherical shell radii ---
    # R_outer = sphere radius + cap thickness (the film is deposited ON the sphere)
    R_inner = sphere_diameter / 2
    R_outer = R_inner + cap_thickness

    # Adaptive mesh resolution strategy:
    #   d <= 1.5 µm : ~18 nm cells (resolves exchange length ~5 nm reasonably well)
    #   d >  1.5 µm : fixed n=110 cells — keeps total count ~0.7 M regardless of diameter.
    #
    # Resulting cell sizes at fixed n=110:
    #   3 µm → ~28 nm | 5 µm → ~47 nm | 8 µm → ~74 nm | 10 µm → ~92 nm | 20 µm → ~183 nm
    #
    # ⚠️  At 20 µm the cell (~183 nm) greatly exceeds the exchange length (~5 nm) and the
    # cap thickness (60 nm). Exchange interactions are underresolved — treat results as a
    # mean-field / Stoner-Wohlfarth estimate, not a fully converged micromagnetic solution.
    # If OOMMF exits with a memory error, reduce n to 80 or 60 for the 20 µm case.
    n_val = int(round((2 * R_outer) / 18e-9)) if sphere_diameter <= 1.5e-6 else 110

    # Upper half-space only (z >= 0) — models just the cap, not the full sphere
    region = df.Region(p1=(-R_outer, -R_outer,       0),
                       p2=( R_outer,  R_outer,  R_outer))
    mesh   = df.Mesh(region=region, n=(n_val, n_val, n_val))

    print(f"\n[RUNNING {d_idx+1}/{len(diameters)}] {safe_name} | Mesh: {n_val}³")

    # --- Spatial field functions assigned cell-by-cell ---

    def Ms_mask(pos):
        """Return Ms inside the hemispherical shell, 0 elsewhere."""
        r = np.linalg.norm(pos)
        return Ms if (R_inner <= r <= R_outer and pos[2] >= 0) else 0

    def u_func(pos):
        """
        Easy-axis direction per cell.
        Radial: axis points along the local outward surface normal.
        Uniaxial_Vertical: axis fixed along global z for all cells.
        Falls back to (0,0,1) at the origin to avoid division by zero.
        """
        if mode == "Radial":
            r = np.linalg.norm(pos)
            return (pos[0]/r, pos[1]/r, pos[2]/r) if r != 0 else (0, 0, 1)
        return (0, 0, 1)  # Uniaxial_Vertical

    # --- Build micromagnetic system ---
    system = mm.System(name=safe_name)
    system.energy = (
        mm.Exchange(A=A)
        + mm.Demag()
        + mm.UniaxialAnisotropy(
            K=Ku,  # Uniform L10 anisotropy — no spatial distribution needed
            u=df.Field(mesh, nvdim=3, value=u_func)
        )
        + mm.Zeeman(H=(0, 0, 0))  # Field updated at each step below
    )

    # LLG dynamics terms — required by TimeDriver, not used by MinDriver.
    # Precession drives magnetisation around the effective field;
    # Damping dissipates energy and allows convergence toward equilibrium.
    if USE_DYNAMIC:
        system.dynamics = (
            mm.Precession(gamma0=mm.consts.gamma0)
            + mm.Damping(alpha=alpha)
        )

    # Initialise magnetisation uniformly along +z
    system.m = df.Field(mesh, nvdim=3, value=(0, 0, 1),
                        norm=df.Field(mesh, nvdim=1, value=Ms_mask))

    # --- Volume sanity check: compare analytical vs discretised shell volume ---
    # Discrepancy arises from the staircase approximation of the curved boundary
    v_theory = (2/3) * np.pi * (R_outer**3 - R_inner**3)
    v_mesh   = (system.m.norm.integrate() / Ms).item()
    print(f"  Vol error: {abs(v_mesh - v_theory) / v_theory * 100:.2f}%")

    # --- Field sweep ---
    hyst_data = []
    driver = oc.TimeDriver() if USE_DYNAMIC else oc.MinDriver()

    for s_idx, B_z in enumerate(B_fields):
        try:
            # Small By tilt prevents the solver getting stuck at a saddle point
            system.energy.zeeman.H = (0, B_tilt / mu0, B_z / mu0)

            if USE_DYNAMIC:
                # T=T activates OOMMF's stochastic thermal field (Langevin dynamics)
                driver.drive(system, t=run_time, n=1, T=T, n_threads=8)
            else:
                # MinDriver minimises total energy — equivalent to 0 K, no dynamics
                driver.drive(system, n_threads=8,
                             stopping_mxHxm=2e4,          # Convergence criterion
                             total_iteration_limit=15000)  # Safety cap on iterations

            # Normalised average Mz over magnetic material only
            m_norm_int = system.m.norm.integrate().item()
            mz_avg     = (system.m.z.integrate().item() / m_norm_int) if m_norm_int > 0 else 0
            hyst_data.append(mz_avg)

            # Progress log every 40 steps
            if s_idx % 40 == 0:
                print(f"   Step {s_idx+1}/242 | B: {B_z:6.2f} T | Mz/Ms: {mz_avg:8.4f}")

        except Exception as e:
            # On OOMMF crash: carry forward last known value to maintain array length
            fallback = hyst_data[-1] if hyst_data else 0
            hyst_data.append(fallback)
            print(f"   [!] Step {s_idx+1} failed ({e}), using fallback value.")

    # --- Save per-cap results ---
    if len(hyst_data) == len(B_fields):
        df_out = pd.DataFrame({'B_ext (T)': B_fields, 'Mz/Ms': hyst_data})
        df_out.to_csv(checkpoint, index=False)
        successful_runs[safe_name] = hyst_data
        print(f"  >> Saved: {checkpoint}")
    else:
        print(f"  !! ERROR: Length mismatch for {safe_name}. Check for OOMMF crashes.")

    oc.delete(system)  # Free OOMMF resources before next run


# ===========================================================================
# 3. FINAL SUMMARY & EXPORTS
# ===========================================================================

if successful_runs:
    master_results = pd.DataFrame({'B_ext (T)': B_fields})
    plt.figure(figsize=(12, 8))

    for key, data in successful_runs.items():
        master_results[key] = data
        plt.plot(B_fields, data, label=key)

    # Combined master CSV — one column per diameter
    master_csv = f"MASTER_DATA_L10_{mode}_{T_label}_{timestamp}.csv"
    master_results.to_csv(os.path.join(OUTPUT_FOLDER, master_csv), index=False)

    plt.xlabel(r"$\mu_0 H$ (T)")
    plt.ylabel(r"$M_z / M_s$ (normalised)")
    plt.title(f"Pure L10 FePt | {mode} | {solver_label} | {timestamp}")
    plt.axhline(0, color='black', lw=0.5)
    plt.axvline(0, color='black', lw=0.5)
    plt.grid(True, ls=':', alpha=0.6)
    plt.legend(fontsize='x-small', ncol=2)

    plot_name = f"SUMMARY_PLOT_L10_{mode}_{T_label}_{timestamp}.svg"
    plt.savefig(os.path.join(OUTPUT_FOLDER, plot_name))
    plt.show()

print(f"\nAll tasks complete. Files saved in: {OUTPUT_FOLDER}")
